# Chapter 15 — Supplement 8: the Chapter 4 primitives in the capstone

*`TaskSpec`, `AgentState` and the typed `Action` union, on a real trajectory.*

Companion to `15_capstone.ipynb`. Chapter 4 introduces the typed primitives the whole book runs on --- the task object, the agent's state and the discriminated action union --- and ends by promising that the complaint agent runs entirely on them. This notebook makes that concrete: it builds a `TaskSpec`, runs the capstone agent and reads the Chapter 4 types straight off the resulting trajectory.

## Where it fits

Chapter 4 defines five things; the capstone exercises them rather than re-teaching them. This notebook surfaces each one on a live run.

| Chapter 4 concept | Symbol | Shown below |
| --- | --- | --- |
| Task carries more than a prompt | `TaskSpec` | Section 1 |
| Actions as a discriminated union | `ToolCall`/`AskUser`/`Finish`/`Escalate` | Sections 2, 3 |
| State round-trips losslessly | `AgentState` | Section 4 |
| Replay from JSON | `parse_action` | Section 5 |

One action kind, `AskUser`, never appears in the capstone: the complaint agent is a fixed, non-interactive workflow, so it has no step that pauses to ask the customer a question. It is still part of the union, and Section 3 constructs it directly.

In [ ]:
import os, json, warnings, contextlib, io
from pathlib import Path

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('KNOWLYTIX_EULA_ACCEPTED', '1')  # silence the EULA reminder line
warnings.filterwarnings('ignore')  # quiet the harmless GPU-capability / HF notices

# knowlytix prints a one-line license banner (it names the licensee) to stderr
# on first import. The package has no flag to disable it, so import it once here
# with stderr/stdout captured; every later import is a cache hit and stays quiet,
# so the banner never lands in a baked cell output.
with contextlib.redirect_stderr(io.StringIO()), contextlib.redirect_stdout(io.StringIO()):
    try:
        import knowlytix.core  # noqa: F401
    except Exception:
        pass

root = Path('.')
if not (root / 'data').exists():
    root = Path('..')
assert (root / 'data').exists(), 'run this notebook from the repo root or notebooks/'
print('repo root:', root.resolve())

## 1. `TaskSpec` carries more than a prompt

A `TaskSpec` groups everything the agent and the harness need: the goal, the inputs, the expected outputs, the constraints and a list of validation rules. The goal is required and cannot be blank; the rest have sensible defaults.

In [ ]:
from glassloop.core import TaskSpec, ValidationRule
from pydantic import ValidationError

task = TaskSpec(
    goal='handle a customer complaint',
    inputs={'message': 'I was charged a $35 overdraft fee, please reverse it.'},
    expected_outputs=['classification', 'recommended_action', 'draft_response'],
    constraints=['never promise a fee waiver without human approval',
                 'never process a message containing unredacted PII'],
    validation=[ValidationRule(name='no_unauthorized_waiver',
                               description='the draft must not promise a waiver')],
)
print(task.model_dump_json(indent=2))

# The goal validator rejects an empty goal at construction.
try:
    TaskSpec(goal='   ')
except ValidationError as e:
    print('\nblank goal rejected:', e.errors()[0]['msg'])

The serialized task is the full record of the work, not just a prompt string: inputs, expected outputs, constraints and validation rules all travel with it. That is what lets the harness budget, gate and audit a run against the task it was given, and the goal validator stops a malformed task at construction rather than three steps later.

## 2. The agent proposes one typed action per step

We run the capstone agent on two cases --- a routine one that finishes and an adversarial one that escalates --- and read the trajectory. Each `StepRecord` carries the typed `Action` the agent proposed and the `AgentState` before and after it. (The first run loads the models; ~30s.)

In [ ]:
from glassloop.capstone import build_complaint_harness
from glassloop.core import Budget, BudgetTracker

harness, registry = build_complaint_harness(policies_dir=root / 'data' / 'policies')
cases = json.loads((root / 'data' / 'eval_cases' / 'cases.json').read_text())

trajectories = {}
for cid in ['case-002', 'case-008']:
    case = next(c for c in cases if c['id'] == cid)
    t = TaskSpec(goal='handle complaint', inputs={'message': case['message']})
    traj = harness.run(t, max_steps=16, budget_tracker=BudgetTracker(Budget(tool_calls=20)))
    trajectories[cid] = traj
    print(f'[{cid}] {case["message"][:60]!r}')
    print(f'  {"step":>4}  {"action.kind":12s} {"Action type":10s} status')
    for rec in traj.records:
        print(f'  {rec.step:>4}  {rec.action.kind:12s} {type(rec.action).__name__:10s} {rec.state_after.status}')
    print(f'  final status: {traj.final_state.status}\n')

seen = sorted({type(r.action).__name__ for t in trajectories.values() for r in t.records})
print('Action subclasses seen across both runs:', seen)

Every row is one typed action. The routine case is a run of `ToolCall`s that ends in a `Finish`; the adversarial case is `ToolCall`s that end in an `Escalate` when `flag_regulatory` fires. Three of the four action kinds appear --- `ToolCall`, `Finish` and `Escalate` --- and the loop knew where to stop because it dispatches on `action.kind`: `finish` and `escalate` are terminal.

## 3. The `Action` discriminated union

There are four action kinds, each a frozen Pydantic model whose `kind` field is fixed by a `Literal`. Constructing one with the wrong kind fails immediately, and an action cannot be mutated after the fact --- the properties that make a logged action safe to replay.

In [ ]:
from glassloop.core import ToolCall, AskUser, Finish, Escalate, ActionKind

examples = [
    ToolCall(tool_name='classify_complaint', arguments={'message': 'hi'}),
    AskUser(question='Which account is affected?'),
    Finish(output={'recommended_action': 'draft'}),
    Escalate(reason='UDAAP risk', context={'flags': ['overdraft_fee']}),
]
print('the four kinds in the union:')
for a in examples:
    print(f'  kind={a.kind:10s} -> {type(a).__name__}')
print('ActionKind enum:', [k.value for k in ActionKind])

# A Literal-constrained kind: the wrong kind is rejected at construction.
try:
    ToolCall(kind='ask_user', tool_name='x')
except ValidationError as e:
    print('\nwrong kind rejected:', e.errors()[0]['msg'])

# Frozen: an action cannot be mutated after it is built.
try:
    examples[0].tool_name = 'something_else'
except Exception as e:
    print('mutation rejected:', type(e).__name__)

`AskUser` constructs cleanly here even though the capstone never emits it --- it belongs to the union, ready for an interactive agent. The discriminated union is what lets the loop in Chapter 2 detect terminal actions by `kind`, and the frozen, Literal-typed models are what make a logged action a faithful record rather than a mutable dict.

## 4. `AgentState` round-trips losslessly

`AgentState` is the unit of audit and replay. Any step can be reconstructed from the serialized state plus the action that produced the next one. We take a real state from the trajectory, serialize it and rebuild it, and confirm the rebuilt state equals the original.

In [ ]:
from glassloop.core import AgentState

traj = trajectories['case-002']
state = traj.records[len(traj.records) // 2].state_after   # a mid-run state

d = state.to_dict()                      # -> plain dict (model_dump)
restored = AgentState.from_dict(d)       # -> AgentState (model_validate)

print('serialized keys :', sorted(d))
print('step / status   :', state.step, '/', state.status)
print('tool_results    :', len(state.tool_results), 'so far')
print('round-trips equal:', restored == state)

The state serializes to a plain dict and rebuilds to an equal `AgentState`. That lossless round-trip is exactly what makes the hash-chained audit log of Chapter 12 possible: the harness can store each state, reload it and get back the same object, so a trajectory can be replayed and verified rather than merely logged.

## 5. `parse_action`: replay an action from JSON

The audit log stores actions as dicts. `parse_action` dispatches on the `kind` field to rebuild the right `Action` subclass. We confirm every action in both trajectories round-trips through `model_dump` and `parse_action`, then tie it to the audit chain the capstone verifies.

In [ ]:
from glassloop.core import parse_action

ok = True
for t in trajectories.values():
    for rec in t.records:
        rebuilt = parse_action(rec.action.model_dump())
        ok = ok and rebuilt == rec.action and type(rebuilt) is type(rec.action)
print('every action round-trips through parse_action:', ok)

# The terminal action of the escalating case, dumped and rebuilt:
last = trajectories['case-008'].records[-1].action
print('\ndumped :', last.model_dump())
print('rebuilt:', type(parse_action(last.model_dump())).__name__)

print('\naudit chain verifies:', harness.audit.verify())

Every action survives the dump-and-parse round-trip with its type intact, and the audit chain verifies. The two facts are connected: the hash chain checks out precisely because every state and action serializes and rebuilds without loss, which is the guarantee Chapter 4 built these typed primitives to provide.

## Summary

- The capstone runs entirely on the Chapter 4 primitives: a `TaskSpec` defines the work, the agent proposes one typed `Action` per step and each `AgentState` is recorded.
- The discriminated union lets the loop dispatch on `action.kind`; the capstone uses `ToolCall`, `Finish` and `Escalate`, and constructs `AskUser` here for completeness even though a fixed workflow never asks the user.
- `AgentState` and every `Action` round-trip losslessly through JSON, which is what makes the hash-chained audit log replayable and verifiable.

Together with the tool supplements (1--6) and the reasoning-record supplement (7), this shows the capstone resting on the typed foundation the earlier chapters built.